MODEL TRAINING AND EVALUATION

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
#loading the csv file
df_all = pd.read_csv("merged_dataset.csv")
df_all.head()

,text,label
0,didnt feel humiliated,sadness
1,go feeling hopeless damned hopeful around some...,sadness
2,im grabbing minute post feel greedy wrong,anger
3,ever feeling nostalgic fireplace know still pr...,love
4,feeling grouchy,anger


In [2]:
df_all['label'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

There are total 6 emotions in our train data

In [3]:
#Loading the test data
df_test= pd.read_csv("testpp.csv")
df_test.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


In [4]:
#Mapping the labels same as the train data
df_test['label']= df_test['label'].map({0:'sadness', 1:'joy', 2:'love', 3:'anger', 4:'fear', 5:'surprise'})
df_test.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


In [5]:
print(df_test['label'].unique())
print(df_test.dtypes)

['sadness' 'joy' 'fear' 'anger' 'love' 'surprise']
text     object
label    object
dtype: object


There are also total 6 emotions in test data same as train data

In [6]:
df_test.isnull().sum()

text     0
label    0
dtype: int64

there are no null values in test data 

In [7]:
#Preprocessing the test data same as train data
import re

def clean_text(text):  #making a function for the preprocessing of the text
    text = text.lower()  # lowercasing the text
    text = re.sub(r'http\S+|www\S+', '', text)  # removing URLs
    text = re.sub(r'@\w+', '', text)  # removing mentions
    text = re.sub(r'#\w+', '', text)  # removing hashtags
    text = re.sub(r'[^a-z\s]', '', text)  # keeping only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # removing extra spaces
    return text

df_test['text'] = df_test['text'].apply(clean_text)
df_test.head(10)

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness
5,i cant walk into a shop anywhere where i do no...,fear
6,i felt anger when at the end of a telephone call,anger
7,i explain why i clung to a relationship with a...,joy
8,i like to have the same breathless feeling as ...,joy
9,i jest i feel grumpy tired and pre menstrual w...,anger


In [8]:
#Removing duplicates
df_test.drop_duplicates(subset='text', inplace=True)
df_test.head(10)

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness
5,i cant walk into a shop anywhere where i do no...,fear
6,i felt anger when at the end of a telephone call,anger
7,i explain why i clung to a relationship with a...,joy
8,i like to have the same breathless feeling as ...,joy
9,i jest i feel grumpy tired and pre menstrual w...,anger


In [9]:
#Removing stopwords
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

df_test['text'] = df_test['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
df_test.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arora\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,label
0,im feeling rather rotten im ambitious right,sadness
1,im updating blog feel shitty,sadness
2,never make separate ever want feel like ashamed,sadness
3,left bouquet red yellow tulips arm feeling sli...,joy
4,feeling little vain one,sadness


Stopwords like in, a, is, etc are removed to make the text short and crisp for the model to understand

In [10]:
#lemmatization (Reducing the words to their dictionary form)
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
df_test['label'] = df_test['label'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
df_test.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arora\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\arora\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,text,label
0,im feeling rather rotten im ambitious right,sadness
1,im updating blog feel shitty,sadness
2,never make separate ever want feel like ashamed,sadness
3,left bouquet red yellow tulips arm feeling sli...,joy
4,feeling little vain one,sadness


Saving the preprocessed test data

In [11]:
df_test.to_csv("test_final.csv", index=False, encoding="utf-8")

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
#loading the train and test datasets
df_all = pd.read_csv("merged_dataset.csv") 
df_test= pd.read_csv("test_final.csv")

In [13]:
df_all.columns


Index(['text', 'label'], dtype='object')

Training different models
1. Tf-Idf+ Linear SVC

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

#Splitting df_all into train and validation data
X_train, X_val, y_train, y_val = train_test_split(
    df_all['text'], df_all['label'],
    test_size=0.2,       # splitting the data such that 20% of data goes as validation data and 80% as train data
    random_state=42,
    stratify=df_all['label'] #splitting the data such that same proportion of emotions data goes to both test and train data
)

# Handling missing values in text column
X_train = X_train.fillna("")
X_val   = X_val.fillna("")
df_test['text'] = df_test['text'].fillna("")

# Encoding labels using label encoder for the model to understand
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(df_test['label'])

# TF-IDF vectorization (fitting only on train)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(df_test['text'])

# Defining evaluation function for logistic regression and SVC linear to work
def evaluate_model(name, model):
    model.fit(X_train_tfidf, y_train_enc)
    print(f"\n{name} Results:")
    
    for split_name, X_split, y_split in [
        ("Train", X_train_tfidf, y_train_enc),
        ("Validation", X_val_tfidf, y_val_enc),
        ("Test", X_test_tfidf, y_test_enc)
    ]:
        y_pred = model.predict(X_split)
        acc = accuracy_score(y_split, y_pred)
        print(f"{split_name} Accuracy: {acc:.4f}")
    
    print("\nClassification Report (Test):")
    print(classification_report(
        y_test_enc, 
        model.predict(X_test_tfidf),
        target_names=le.classes_
    ))

# Linear SVC
svc = LinearSVC()
evaluate_model("Linear SVC", svc)



Linear SVC Results:
Train Accuracy: 0.9959
Validation Accuracy: 0.9195
Test Accuracy: 0.8940

Classification Report (Test):
              precision    recall  f1-score   support

       anger       0.89      0.89      0.89       275
        fear       0.91      0.82      0.86       224
         joy       0.91      0.92      0.92       695
        love       0.77      0.81      0.79       159
     sadness       0.93      0.94      0.93       581
    surprise       0.70      0.67      0.68        66

    accuracy                           0.89      2000
   macro avg       0.85      0.84      0.85      2000
weighted avg       0.89      0.89      0.89      2000



99.59% of training accuracy suggest that the model fits the training data almost perfectly.
 
Validation accuracy (91.95%) and test accuracy (89.40%) are slightly lower than training, but still strong,This means the model generalizes well to unseen data.

The small drop from validation to test performance suggests the model’s performance is consistent and not heavily overfitting.

2. Tf-Idf+Logistic Regression

In [15]:
#Logistic Regression
lr = LogisticRegression(max_iter=200, solver='liblinear')
evaluate_model("Logistic Regression", lr)

C:\Users\arora\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



Logistic Regression Results:
Train Accuracy: 0.9380
Validation Accuracy: 0.8804
Test Accuracy: 0.8650

Classification Report (Test):
              precision    recall  f1-score   support

       anger       0.92      0.77      0.84       275
        fear       0.90      0.80      0.85       224
         joy       0.83      0.97      0.89       695
        love       0.84      0.57      0.68       159
     sadness       0.88      0.95      0.92       581
    surprise       0.96      0.39      0.56        66

    accuracy                           0.86      2000
   macro avg       0.89      0.74      0.79      2000
weighted avg       0.87      0.86      0.86      2000



Training accuracy (93.80%), means The model learns patterns well but not as tightly as Linear SVC.

Validation accuracy (88.04%) and test accuracy (86.50%) means Good generalization, but performance drops more from training compared to Linear SVC.

The slightly larger gap between training and test scores suggests less overfitting risk than Linear SVC, but also lower maximum performance.



Linear SVC outperforms Logistic Regression in all accuracy metrics by ~2–3%, showing it captures patterns better for this task.

3. Naive Baye's

In [16]:
#trying naive bayes
from sklearn.naive_bayes import MultinomialNB
# Naive Bayes
nb = MultinomialNB()
evaluate_model("Naive Bayes", nb)



Naive Bayes Results:
Train Accuracy: 0.8239
Validation Accuracy: 0.7505
Test Accuracy: 0.7500

Classification Report (Test):
              precision    recall  f1-score   support

       anger       0.96      0.49      0.65       275
        fear       0.88      0.51      0.65       224
         joy       0.70      0.99      0.82       695
        love       1.00      0.13      0.22       159
     sadness       0.75      0.94      0.83       581
    surprise       1.00      0.03      0.06        66

    accuracy                           0.75      2000
   macro avg       0.88      0.51      0.54      2000
weighted avg       0.80      0.75      0.71      2000



training accuracy (82.39%) means The model captures basic patterns but is less powerful than Linear SVC or Logistic Regression.

Validation accuracy (75.05%) and test accuracy (75.00%) means Lower performance compared to other models, indicating that it struggles to capture the complexity of the dataset.

Very small gap between training, validation, and test accuracies suggests the model generalizes consistently, but its overall predictive capacity is limited.

In [17]:
%pip install tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import numpy as np

Note: you may need to restart the kernel to use updated packages.


C:\Users\arora\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\arora\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\arora\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framewo

4. LSTM model

In [18]:
# Tokenizing text
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=max_len)
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(df_test['text']), maxlen=max_len)

# Building LSTM model
model = Sequential()
model.add(Embedding(max_words, 128, input_length=max_len))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(len(le.classes_), activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# Training model
model.fit(X_train_seq, y_train_enc, validation_data=(X_val_seq, y_val_enc), epochs=5, batch_size=64)
# Evaluating
train_acc = model.evaluate(X_train_seq, y_train_enc, verbose=0)[1]
val_acc   = model.evaluate(X_val_seq, y_val_enc, verbose=0)[1]
test_acc  = model.evaluate(X_test_seq, y_test_enc, verbose=0)[1]

print(f"LSTM Results:\nTrain Accuracy: {train_acc:.4f}\nValidation Accuracy: {val_acc:.4f}\nTest Accuracy: {test_acc:.4f}")




C:\Users\arora\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 367ms/step - accuracy: 0.5029 - loss: 1.3061 - val_accuracy: 0.7833 - val_loss: 0.6971
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 72s 361ms/step - accuracy: 0.8910 - loss: 0.3356 - val_accuracy: 0.9095 - val_loss: 0.2540
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 87s 384ms/step - accuracy: 0.9577 - loss: 0.1223 - val_accuracy: 0.9148 - val_loss: 0.2289
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 93s 439ms/step - accuracy: 0.9761 - loss: 0.0704 - val_accuracy: 0.9045 - val_loss: 0.2756
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 80s 398ms/step - accuracy: 0.9848 - loss: 0.0485 - val_accuracy: 0.9195 - val_loss: 0.2605
LSTM Results:
Train Accuracy: 0.9915
Validation Accuracy: 0.9195
Test Accuracy: 0.9055


Training accuracy (99.15%) means the model learns the training data extremely well.

Validation accuracy (91.95%) and test accuracy (90.55%) means high and consistent performance on unseen data, showing excellent generalization.

The gap between training and test accuracy (~8.6%) is slightly larger than Logistic Regression but similar to Linear SVC, which is expected for deep learning models due to their higher capacity.

###Summary

Linear SVC: High accuracy (~89.4%), strong generalization, slightly better than Logistic Regression.
Logistic Regression: Balanced performance (~86.5% test accuracy), simple and less prone to overfitting.
Naive Bayes: Fast and consistent but lowest accuracy (~75%), good as a quick baseline.
LSTM: Best performance (~90.7% test accuracy), captures context and word order effectively, ideal for text data.

Conclusion:
LSTM is the most suitable model for this project due to its superior accuracy and ability to understand sequential patterns in text. 
This model, along with its tokenizer and label encoder, will be saved for deployment.

In [19]:
##Saving the model
import pickle
import joblib

#Saving the Tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

#Saving the LabelEncoder
joblib.dump(le, "label_encoder.pkl")

#Saving the LSTM Model
model.save("lstm_emotion_model.h5")

print(" Model, tokenizer, and label encoder saved successfully!")


✅ Model, tokenizer, and label encoder saved successfully!


##Future Scope of Project

Dataset Expansion: Add 2 more datasets and unify emotion labels.  
    
Model Enhancement: Try BERT/DistilBERT, compare with LSTM, use ensemble methods, and fine-tune hyperparameters.   
    
Deployment: Build Flask/FastAPI API, integrate with a mental health chatbot, and create a Streamlit dashboard.  
    
Evaluation: Test on unseen datasets, track precision/recall per class, and conduct human evaluation.  
    
Research: Document results, compare models, and aim for research paper/publication.
